# SantaCoder tokenization and prompt design

This notebook is meant to teach the basic interaction loop with TransformerLens on SantaCoder before we try to discover induction heads.

The main goals are:

- understand how SantaCoder tokenizes code-like inputs;
- learn the minimum TransformerLens API needed for circuit work;
- inspect the activation cache produced by `run_with_cache`;
- define prompt families we will reuse in later notebooks.

If this notebook feels repetitive, that is intentional. The point is to build intuition for how text becomes tokens, how tokens become activations, and how we can inspect those activations layer by layer.

## Setup

In TransformerLens, the main object is `HookedTransformer`. It wraps a pretrained model and exposes useful helper methods such as:

- `to_tokens`: convert text to token IDs;
- `to_str_tokens`: convert text to token strings;
- `run_with_cache`: run the model and save intermediate activations;
- `cache["pattern", layer]`: access the attention pattern of a given layer.

For SantaCoder, we apply the compatibility patch first, then load the model on CPU.

In [1]:
import torch
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.patch_santacoder import ensure_santacoder_transformers_compat
from transformer_lens import HookedTransformer

ensure_santacoder_transformers_compat()

model = HookedTransformer.from_pretrained(
    "santacoder",
    device="cpu"
)

print("Model loaded")
print("device:", model.cfg.device)
print("dtype:", next(model.parameters()).dtype)
print("n_layers:", model.cfg.n_layers)
print("n_heads:", model.cfg.n_heads)
print("d_model:", model.cfg.d_model)
print("tokenizer:", model.cfg.tokenizer_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model santacoder into HookedTransformer
Model loaded
device: cpu
dtype: torch.float32
n_layers: 24
n_heads: 16
d_model: 2048
tokenizer: bigcode/santacoder


## Helper functions

These helpers are intentionally simple. They are the kind of small utilities you will keep reusing during exploratory circuit work.

In [2]:
def inspect_text(text, prepend_bos=True):
    tokens = model.to_tokens(text, prepend_bos=prepend_bos)
    token_ids = tokens[0].tolist()
    str_tokens = model.to_str_tokens(text, prepend_bos=prepend_bos)
    rows = []
    for position, (token_id, token_str) in enumerate(zip(token_ids, str_tokens)):
        rows.append({
            "position": position,
            "token_id": token_id,
            "token_str": repr(token_str),
        })
    return pd.DataFrame(rows)


def top_next_token_predictions(text, k=10, prepend_bos=True):
    tokens = model.to_tokens(text, prepend_bos=prepend_bos)
    logits = model(tokens)
    last_logits = logits[0, -1]
    probs = last_logits.softmax(dim=-1)
    top_probs, top_ids = torch.topk(probs, k)
    rows = []
    for rank, (token_id, prob) in enumerate(zip(top_ids.tolist(), top_probs.tolist()), start=1):
        rows.append({
            "rank": rank,
            "token_id": token_id,
            "token_str": repr(model.to_single_str_token(token_id)),
            "probability": prob,
        })
    return pd.DataFrame(rows)


def first_layer_attention_table(text, head=0, prepend_bos=True):
    tokens = model.to_tokens(text, prepend_bos=prepend_bos)
    str_tokens = model.to_str_tokens(text, prepend_bos=prepend_bos)
    _, cache = model.run_with_cache(tokens)
    pattern = cache["pattern", 0][0, head]
    df = pd.DataFrame(pattern.detach().cpu().numpy())
    df.index = [f"q{idx}:{tok!r}" for idx, tok in enumerate(str_tokens)]
    df.columns = [f"k{idx}:{tok!r}" for idx, tok in enumerate(str_tokens)]
    return df.round(3)


## Probe 1: a small code snippet

Start with a tiny Python-like function. Notice how SantaCoder merges spaces with nearby tokens and how newline-plus-indentation can become its own token fragment. This matters later because induction or structural behavior happens over tokens, not over human-visible characters.

In [4]:
simple_code = "def add(a, b):\n    return a + b\n"

inspect_text(simple_code)

,position,token_id,token_str
0,0,49152,'<|endoftext|>'
1,1,563,'def'
2,2,1030,' add'
3,3,7,'('
4,4,64,'a'
5,5,11,"','"
6,6,329,' b'
7,7,399,'):'
8,8,258,'\n '
9,9,363,' return'


It is also useful to inspect the model's immediate next-token distribution. This helps connect tokenization to actual predictions.

In [5]:
top_next_token_predictions(simple_code, k=10)

,rank,token_id,token_str,probability
0,1,185,'\n',0.876884
1,2,563,'def',0.050555
2,3,2,'#',0.031413
3,4,1018,'print',0.021801
4,5,64,'a',0.003248
5,6,478,'add',0.002348
6,7,1114,"'""""""'",0.001378
7,8,626,'class',0.001056
8,9,843,'assert',0.000843
9,10,302,'if',0.000635


In [3]:
top_next_token_predictions("Sum of one and two and four (1+2+", k=100)

,rank,token_id,token_str,probability
0,1,19,'4',0.946026
1,2,18,'3',0.040894
2,3,17,'2',0.006003
3,4,16,'1',0.003618
4,5,1293,'...',0.000528
...,...,...,...,...
95,96,9671,'***',0.000001
96,97,15760,'xxx',0.000001
97,98,85,'v',0.000001
98,99,73,'j',0.000001


## Probe 2: repeated patterns for induction

Induction heads are easiest to study on prompts with a clear repeated pattern. The classic intuition is:

1. a pattern appears once;
2. the same prefix appears again later;
3. some heads learn to attend from the second occurrence to the first one and copy what came next.

Here we start with code-like repeated identifiers and assignments.

In [7]:
repeated_code = (
    "user_id = fetch_user()\n"
    "log(user_id)\n"
    "\n"
    "user_id = fetch_user()\n"
    "log("
)

inspect_text(repeated_code)

,position,token_id,token_str
0,0,49152,'<|endoftext|>'
1,1,789,'user'
2,2,62,'_'
3,3,301,'id'
4,4,256,' ='
5,5,4097,' fetch'
6,6,62,'_'
7,7,789,'user'
8,8,295,'()'
9,9,185,'\n'


In [8]:
top_next_token_predictions(repeated_code, k=10)

,rank,token_id,token_str,probability
0,1,789,'user',0.994894
1,2,69,'f',0.001991
2,3,4028,'fetch',0.000658
3,4,605,'type',0.000244
4,5,491,'str',0.000204
5,6,307,'get',0.000182
6,7,16,'1',0.000098
7,8,868,'use',0.000073
8,9,84,'u',0.000066
9,10,301,'id',0.000064


## Probe 3: bracket-heavy prompts

Later we will study bracket matching. For now, just inspect how bracket-rich strings are tokenized. In code models, brackets often interact with spaces, commas, colons, and newlines in ways that differ from plain English text.

In [9]:
bracket_prompt = "items = [(x, {y: [z for z in data]}) for x in xs"

inspect_text(bracket_prompt)

,position,token_id,token_str
0,0,49152,'<|endoftext|>'
1,1,2063,'items'
2,2,256,' ='
3,3,6198,' [('
4,4,87,'x'
5,5,11,"','"
6,6,274,' {'
7,7,88,'y'
8,8,25,':'
9,9,436,' ['


In [13]:
top_next_token_predictions(bracket_prompt, k=10)

,rank,token_id,token_str,probability
0,1,408,' for',0.854198
1,2,60,']',0.104445
2,3,62,'_',0.006458
3,4,356,' if',0.005650
4,5,444,'\n ',0.004934
5,6,13,'.',0.001728
6,7,286,'\n ',0.001693
7,8,259,'\n ',0.001324
8,9,2545,'range',0.001176
9,10,16,'1',0.001171


## Probe 4: indentation-sensitive prompts

For later circuit-discovery work on indentation, it helps to see early that newline-plus-spaces often form token pieces. That means indentation depth may be represented partly through token identity and partly through higher-level activations.

In [10]:
indent_prompt = (
    "def outer(x):\n"
    "    if x > 0:\n"
    "        for item in xs:\n"
    "            print(item)\n"
    "    return x\n"
)

inspect_text(indent_prompt)

,position,token_id,token_str
0,0,49152,'<|endoftext|>'
1,1,563,'def'
2,2,10708,' outer'
3,3,7,'('
4,4,87,'x'
5,5,399,'):'
6,6,258,'\n '
7,7,356,' if'
8,8,727,' x'
9,9,805,' >'


## First `run_with_cache` walkthrough

This is the core TransformerLens workflow:

1. choose a prompt;
2. run the model with `run_with_cache`;
3. inspect attention patterns or residual stream activations;
4. form a hypothesis;
5. later, test it with patching or ablation.

For induction work, attention patterns are a natural first object to inspect. The tensor `cache["pattern", layer]` has shape:

- batch
- head
- query position
- key position

So entry `(head, q, k)` tells us how much that head at query position `q` attends to key position `k`.

In [18]:
tokens = model.to_tokens(indent_prompt)
str_tokens = model.to_str_tokens(indent_prompt)
logits, cache = model.run_with_cache(tokens)

print("tokens shape:", tuple(tokens.shape))
print("logits shape:", tuple(logits.shape))
print("layer 0 attention pattern shape:", tuple(cache["pattern", 0].shape))
print("example tokens:", str_tokens)

tokens shape: (1, 28)
logits shape: (1, 28, 49280)
layer 0 attention pattern shape: (1, 16, 28, 28)
example tokens: ['<|endoftext|>', 'def', ' outer', '(', 'x', '):', '\n   ', ' if', ' x', ' >', ' ', '0', ':', '\n       ', ' for', ' item', ' in', ' xs', ':', '\n           ', ' print', '(', 'item', ')', '\n   ', ' return', ' x', '\n']


In [20]:
first_layer_attention_table(indent_prompt, head=0)

,k0:'<|endoftext|>',k1:'def',k2:' outer',k3:'(',k4:'x',k5:'):',k6:'\n ',k7:' if',k8:' x',k9:' >',...,k18:':',k19:'\n ',k20:' print',k21:'(',k22:'item',k23:')',k24:'\n ',k25:' return',k26:' x',k27:'\n'
q0:'<|endoftext|>',1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q1:'def',0.920,0.080,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q2:' outer',0.597,0.112,0.291,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q3:'(',0.788,0.063,0.092,0.057,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q4:'x',0.718,0.055,0.083,0.065,0.080,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q5:'):',0.618,0.059,0.090,0.056,0.083,0.094,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q6:'\n ',0.720,0.037,0.037,0.024,0.029,0.037,0.115,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q7:' if',0.638,0.041,0.051,0.040,0.050,0.058,0.068,0.055,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q8:' x',0.562,0.055,0.089,0.032,0.051,0.077,0.044,0.057,0.033,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
q9:' >',0.445,0.046,0.081,0.048,0.069,0.070,0.071,0.049,0.058,0.062,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


You should not expect head `0.0` to already be an induction head. This table is here to make the object concrete.

What we will eventually look for is attention from later repeated tokens back to earlier matching context, especially near the boundary where the model should copy the continuation from the first occurrence.

## Prompt template candidates

The next notebooks will need prompt families, not just one prompt. We store them in a small dictionary here so the later workflow stays consistent.

These are still exploratory templates. The important part is that each family targets a different structural phenomenon:

- `induction`: repeated code fragments;
- `brackets`: nested delimiters;
- `indentation`: block structure in Python-like code.

In [ ]:
PROMPT_TEMPLATES = {
    "induction_identifier": (
        "user_id = fetch_user()\n"
        "log(user_id)\n"
        "\n"
        "user_id = fetch_user()\n"
        "log("
    ),
    "induction_assignment": (
        "result = parse(data)\n"
        "print(result)\n"
        "\n"
        "result = parse(data)\n"
        "print("
    ),
    "brackets_python": "items = [(x, {y: [z for z in data]}) for x in xs]\nitems[0]",
    "indentation_python": (
        "def outer(x):\n"
        "    if x > 0:\n"
        "        for item in xs:\n"
        "            print(item)\n"
        "    return x\n"
    ),
}

list(PROMPT_TEMPLATES.keys())

## What to take away from this notebook

Before moving on, make sure these ideas feel concrete:

1. tokenization is part of the mechanism, not just preprocessing;
2. `run_with_cache` is the standard entrypoint for exploratory mech interp in TransformerLens;
3. attention patterns are indexed by layer, then head, then query/key positions;
4. good circuit work starts from carefully chosen prompt families.

The next notebook will use these prompt templates to define an induction score and rank candidate heads.